<a href="https://colab.research.google.com/github/pavitrachouhan/lstm-text-generator/blob/main/quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.9 MB/s eta 0:00:00


In [78]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)
print("Groq client initialized successfully.")

Groq client initialized successfully.


In [79]:
def summarize_notes(notes: str) -> str:
    """
    Summarizes a given block of notes using the Groq API.

    Args:
        notes (str): The text content of the notes to be summarized.

    Returns:
        str: The concise summary generated by the LLM.
    """
    prompt = f"""You are an expert summarizer. Please provide a concise summary of the following notes. Focus on key information and main points, keeping the summary brief and to the point.\n\nNotes:\n{notes}\n\nConcise Summary:"""

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="llama-3.1-8b-instant", # Updated to a currently supported model
        temperature=0.5,
        max_tokens=1500,
    )

    summary = chat_completion.choices[0].message.content
    return summary

# Test the function with sample_notes
summary_output = summarize_notes(sample_notes)
print("Generated Summary:")
print(summary_output)

Generated Summary:
**Project Alpha Launch Meeting Summary (October 26, 2023)**

Key Points:

- Development tasks are 95% complete with minor bug fixes remaining (5%).
- Critical bugs will be resolved by October 27th, minor bugs by November 1st.
- Marketing plan focuses on social media campaigns and influencer partnerships, with launch announcement draft approved.
- Comprehensive regression testing is underway, with User Acceptance Testing (UAT) scheduled for October 30th-31st.
- Official launch date is November 5, 2023, pending successful UAT and bug fixes.

Action Items:
- John: Finalize launch checklist by October 28th.
- Jane: Fix critical bugs by October 27th.
- Alex: Finalize social media content and influencer outreach by November 3rd.
- Emily: Oversee UAT and report findings by November 1st.

Next Meeting: November 2, 2023, 10:00 AM EST to review UAT results and final launch readiness.


In [80]:
def generate_quiz_questions(notes: str, difficulty: str) -> str:
    """
    Generates multiple-choice quiz questions from notes using the Groq API,
    adapting complexity based on the specified difficulty.

    Args:
        notes (str): The text content of the notes.
        difficulty (str): The desired difficulty level ('easy', 'medium', 'hard').

    Returns:
        str: A string containing the generated quiz questions.
    """
    prompt = f"""You are an expert quiz generator. Create multiple-choice quiz questions from the following notes.
    The questions should be suitable for a '{difficulty}' difficulty level. Include 4 options (A, B, C, D) and clearly state the correct answer.

    Notes:
    {notes}

    Instructions for output format:
    Each question should be numbered and followed by its options.
    The correct answer should be clearly marked after the options, for example: 'Correct Answer: C) [Option Text]'.

    Please generate at least 3-5 questions.

    Quiz Questions ({difficulty} difficulty):"""

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="llama-3.1-8b-instant",
        temperature=0.7 if difficulty == 'hard' else (0.5 if difficulty == 'medium' else 0.3), # Adjust temperature based on difficulty
        max_tokens=2000,
    )

    quiz_questions = chat_completion.choices[0].message.content
    return quiz_questions

# Test the function with sample_notes and a chosen difficulty
print("\n--- Generating Quiz Questions (Medium Difficulty) ---")
quiz_output = generate_quiz_questions(sample_notes, difficulty='medium')
print(quiz_output)


--- Generating Quiz Questions (Medium Difficulty) ---
Here are 5 medium difficulty quiz questions based on the provided notes:

1. What percentage of development tasks were confirmed complete by John Doe during the October 26, 2023 meeting?
A) 85%
B) 90%
C) 95%
D) 98%

Correct Answer: C) 95%

2. When are the minor bugs identified during internal testing expected to be addressed?
A) End of day, October 27th
B) November 1st
C) November 5, 2023
D) November 10, 2023

Correct Answer: B) November 1st

3. What is the main focus of the revised marketing plan presented by Alex Chen?
A) Email marketing campaigns
B) Social media campaigns and influencer partnerships
C) Paid advertising on Google
D) Content marketing on YouTube

Correct Answer: B) Social media campaigns and influencer partnerships

4. Who is responsible for overseeing User Acceptance Testing (UAT) and reporting findings by November 1st?
A) John Doe
B) Jane Smith
C) Emily White
D) Alex Chen

Correct Answer: C) Emily White

5. What

In [75]:
import re

def parse_quiz_output(raw_quiz_string: str, difficulty: str) -> list:
    """
    Parses the raw text output from the Groq API for quiz questions into a structured format.

    Args:
        raw_quiz_string (str): The raw string containing quiz questions.
        difficulty (str): The difficulty level used to generate the quiz.

    Returns:
        list: A list of dictionaries, where each dictionary represents a structured question.
    """
    structured_questions = []
    # Regex to capture question number, question text, options, and correct answer
    question_pattern = re.compile(
        r'(\d+)\.\s*(.*?)\n'             # Question number and text
        r'([A-D])\)\s*(.*?)\n'           # Option A key and text
        r'([A-D])\)\s*(.*?)\n'           # Option B key and text
        r'([A-D])\)\s*(.*?)\n'           # Option C key and text
        r'([A-D])\)\s*(.*?)\n\n'          # Option D key and text, followed by double newline
        r'Correct Answer:\s*([A-D])\)\s*(.*)' # Correct Answer key and text, changed (.*?) to (.*)
    )

    matches = question_pattern.findall(raw_quiz_string)

    for match in matches:
        # Unpack the matched groups with parentheses for multi-line assignment
        (q_num_str, question_text,
        opt_a_key, opt_a_text,
        opt_b_key, opt_b_text,
        opt_c_key, opt_c_text,
        opt_d_key, opt_d_text,
        correct_key, correct_text) = match

        options = {
            opt_a_key: opt_a_text.strip(),
            opt_b_key: opt_b_text.strip(),
            opt_c_key: opt_c_text.strip(),
            opt_d_key: opt_d_text.strip()
        }

        structured_questions.append({
            "question_number": int(q_num_str),
            "question": question_text.strip(),
            "options": options,
            "correct_answer_key": correct_key.strip(),
            "correct_answer_text": correct_text.strip(),
            "difficulty": difficulty
        })

    return structured_questions

# Test the parsing function with quiz_output and the difficulty used
structured_quiz_questions = parse_quiz_output(quiz_output, difficulty='medium')

print("\n--- Structured Quiz Questions ---")
for q in structured_quiz_questions:
    print(f"Question Number: {q['question_number']}")
    print(f"Question: {q['question']}")
    print(f"Options: {q['options']}")
    print(f"Correct Answer Key: {q['correct_answer_key']}")
    print(f"Correct Answer Text: {q['correct_answer_text']}")
    print(f"Difficulty: {q['difficulty']}")
    print("---------------------------------")


--- Structured Quiz Questions ---
Question Number: 1
Question: What percentage of development tasks are complete as of the meeting on October 26, 2023?
Options: {'A': '90%', 'B': '95%', 'C': '98%', 'D': '99%'}
Correct Answer Key: B
Correct Answer Text: 95%
Difficulty: medium
---------------------------------
Question Number: 2
Question: According to Jane Smith, how many critical bugs need to be resolved by the end of day, October 27th?
Options: {'A': '3', 'B': '5', 'C': '7', 'D': '10'}
Correct Answer Key: A
Correct Answer Text: 3
Difficulty: medium
---------------------------------
Question Number: 3
Question: What is the scheduled date for User Acceptance Testing (UAT)?
Options: {'A': 'October 25-26', 'B': 'October 30-31', 'C': 'November 1-2', 'D': 'November 3-4'}
Correct Answer Key: B
Correct Answer Text: October 30-31
Difficulty: medium
---------------------------------
Question Number: 4
Question: Who is responsible for overseeing UAT and reporting findings by November 1st?
Option

In [81]:
custom_notes = """
Introduction to Statistics
Statistics is the science of analyzing data.
When we have created a model for prediction, we must assess the prediction's reliability. After all, what is a prediction worth, if we cannot rely on it?

Descriptive Statistics
We will first cover some basic descriptive statistics. Descriptive statistics summarizes important features of a data set such as: Count, Sum, Standard Deviation, Percentile, Average, Etc.. It is a way to describe, organize, and summarize data. Common descriptive statistics include measures of central tendency (mean, median, mode) and measures of variability (range, variance, standard deviation).

Inferential Statistics
Inferential statistics makes inferences about populations using data drawn from the population. It uses probability to determine how confident we can be that the conclusions drawn from a sample are true for the entire population. Techniques include hypothesis testing, confidence intervals, and regression analysis. It allows us to generalize from a sample to a larger population.

Key Differences
Descriptive statistics provides summaries about the sample. Inferential statistics makes predictions about the population.

Applications
Statistics is used in various fields like medicine (clinical trials), business (market research), social sciences (surveys), engineering (quality control), and finance (risk assessment).

Importance of Data Collection
Accurate and unbiased data collection is crucial for valid statistical analysis. Poor data collection can lead to misleading conclusions, regardless of the statistical methods applied.
"""

print("--- Summarizing Custom Notes ---")
custom_summary_output = summarize_notes(custom_notes)
print("Generated Custom Summary:")
print(custom_summary_output)

print("\n--- Generating Custom Quiz Questions (Hard Difficulty) ---")
custom_raw_quiz_output = generate_quiz_questions(custom_notes, difficulty='hard')
print(custom_raw_quiz_output)

print("\n--- Structured Custom Quiz Questions (Hard Difficulty) ---")
structured_custom_quiz_questions = parse_quiz_output(custom_raw_quiz_output, difficulty='hard')

for q in structured_custom_quiz_questions:
    print(f"Question Number: {q['question_number']}")
    print(f"Question: {q['question']}")
    print(f"Options: {q['options']}")
    print(f"Correct Answer Key: {q['correct_answer_key']}")
    print(f"Correct Answer Text: {q['correct_answer_text']}")
    print(f"Difficulty: {q['difficulty']}")
    print("---------------------------------")


--- Summarizing Custom Notes ---
Generated Custom Summary:
Here's a concise summary of the notes:

**Statistics Overview**

Statistics is the science of analyzing data to understand its features and make predictions about a larger population.

**Key Concepts**

- Descriptive statistics: summarizes data features (mean, median, mode, standard deviation, etc.) to describe and organize data.
- Inferential statistics: makes inferences about a population using sample data, using techniques like hypothesis testing and regression analysis.

**Key Differences**

- Descriptive statistics provides summaries about the sample.
- Inferential statistics makes predictions about the population.

**Applications and Importance**

Statistics is used in various fields (medicine, business, social sciences, engineering, finance) and accurate data collection is crucial for valid statistical analysis.

--- Generating Custom Quiz Questions (Hard Difficulty) ---
1. What is the primary objective of inferential st

# Task
```python
import re

def parse_quiz_output(raw_quiz_string: str, difficulty: str) -> list:
    """
    Parses the raw text output from the Groq API for quiz questions into a structured format.

    Args:
        raw_quiz_string (str): The raw string containing quiz questions.
        difficulty (str): The difficulty level used to generate the quiz.

    Returns:
        list: A list of dictionaries, where each dictionary represents a structured question.
    """
    structured_questions = []
    # Regex to capture question number, question text, options, and correct answer
    # Adjusted regex for correct_answer_text to be greedy (.*) to capture the full text
    question_pattern = re.compile(
        r'(\d+)\.\s*(.*?)\n'             # Question number and text
        r'([A-D])\)\s*(.*?)\n'           # Option A key and text
        r'([A-D])\)\s*(.*?)\n'           # Option B key and text
        r'([A-D])\)\s*(.*?)\n'           # Option C key and text
        r'([A-D])\)\s*(.*?)\n\n'         # Option D key and text, followed by double newline
        r'Correct Answer:\s*([A-D])\)\s*(.*)' # Correct Answer key and text (using .* for greedy match)
    )

    matches = question_pattern.findall(raw_quiz_string)

    for match in matches:
        # Unpack the matched groups with parentheses for multi-line assignment
        (q_num_str, question_text,
        opt_a_key, opt_a_text,
        opt_b_key, opt_b_text,
        opt_c_key, opt_c_text,
        opt_d_key, opt_d_text,
        correct_key, correct_text) = match

        options = {
            opt_a_key: opt_a_text.strip(),
            opt_b_key: opt_b_text.strip(),
            opt_c_key: opt_c_text.strip(),
            opt_d_key: opt_d_text.strip()
        }

        structured_questions.append({
            "question_number": int(q_num_str),
            "question": question_text.strip(),
            "options": options,
            "correct_answer_key": correct_key.strip(),
            "correct_answer_text": correct_text.strip(),
            "difficulty": difficulty
        })

    return structured_questions

# Test the parsing function with custom_raw_quiz_output and the difficulty used
print("\n--- Structured Custom Quiz Questions (Hard Difficulty) after regex fix ---")
structured_custom_quiz_questions = parse_quiz_output(custom_raw_quiz_output, difficulty='hard')

for q in structured_custom_quiz_questions:
    print(f"Question Number: {q['question_number']}")
    print(f"Question: {q['question']}")
    print(f"Options: {q['options']}")
    print(f"Correct Answer Key: {q['correct_answer_key']}")
    print(f"Correct Answer Text: {q['correct_answer_text']}")
    print(f"Difficulty: {q['difficulty']}")
    print("---------------------------------")

```

In [82]:
print("\n--- Structured Custom Quiz Questions (Hard Difficulty) after regex fix ---")
structured_custom_quiz_questions = parse_quiz_output(custom_raw_quiz_output, difficulty='hard')

for q in structured_custom_quiz_questions:
    print(f"Question Number: {q['question_number']}")
    print(f"Question: {q['question']}")
    print(f"Options: {q['options']}")
    print(f"Correct Answer Key: {q['correct_answer_key']}")
    print(f"Correct Answer Text: {q['correct_answer_text']}")
    print(f"Difficulty: {q['difficulty']}")
    print("---------------------------------")


--- Structured Custom Quiz Questions (Hard Difficulty) after regex fix ---


In [77]:
import json

def save_quiz_to_json(quiz_data: list, filename: str):
    """
    Saves a list of structured quiz questions to a JSON file.

    Args:
        quiz_data (list): A list of dictionaries, where each dictionary represents a structured question.
        filename (str): The name of the file to save the JSON data to.
    """
    with open(filename, 'w') as f:
        json.dump(quiz_data, f, indent=4)
    print(f"Structured quiz questions saved to {filename}")

# Save structured_quiz_questions (medium difficulty) to a JSON file
save_quiz_to_json(structured_quiz_questions, 'structured_quiz_questions_medium.json')

# Save structured_custom_quiz_questions (hard difficulty) to a JSON file
save_quiz_to_json(structured_custom_quiz_questions, 'structured_quiz_questions_hard.json')

print("All structured quiz data saved successfully.")

Structured quiz questions saved to structured_quiz_questions_medium.json
Structured quiz questions saved to structured_quiz_questions_hard.json
All structured quiz data saved successfully.


In [1]:
import json

def load_quiz_from_json(filename: str) -> list:
    """
    Loads structured quiz questions from a JSON file.

    Args:
        filename (str): The path to the JSON file.

    Returns:
        list: A list of dictionaries, where each dictionary represents a structured question.
    """
    with open(filename, 'r') as f:
        quiz_data = json.load(f)
    print(f"Structured quiz questions loaded from {filename}")
    return quiz_data
